# 04. Add conditions while retaining a normal anchor

We will continue representation learning with additional conditions, replay normal data, and use validation labels to compare frozen feature readouts.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

In [ ]:
from aaai_gait.workflow import prepare_corpus
x, meta, provenance = prepare_corpus(settings)
print('Window tensor:', x.shape, '[windows, frames, selected joints, XY]')
display(meta.groupby(['split','label']).agg(windows=('sequence_id','size'), sequences=('sequence_id','nunique'), sources=('source_id','nunique')))

## 1. Define the progressive curriculum

The initial model learns only normal training sources. A copy then learns from normal plus Parkinson's data, followed by stroke, cerebral palsy, and myopathic data. Each stage keeps all earlier conditions. Every training batch includes normal replay and samples sources within labels. Labels regulate sampling, but the JEPA and VICReg losses themselves use no class targets.

The order is a research hypothesis. It is not justified by a presumed severity ranking. Compare it with pooled training and reversed order before claiming a curriculum advantage.

![Curriculum](images/curriculum.svg)

In [ ]:
from aaai_gait.workflow import normal_anchor, progressive_model
anchor,normal_history=normal_anchor(x,meta,provenance,settings)
before={k:v.clone() for k,v in anchor.state_dict().items()}
adapted,stage_history=progressive_model(anchor,x,meta,provenance,settings)
assert all(torch.equal(v,before[k]) for k,v in anchor.state_dict().items())
display(pd.DataFrame(stage_history)[['stage','epoch','prediction','vicreg','embedding_std']])
print('The normal anchor is unchanged.')

## 2. Train a separate condition readout

A linear probe turns a complete observed window into five condition probabilities. The encoder stays frozen. Standardization and logistic-regression weights use training data only. We compare three regularization strengths on validation sources, aggregate overlapping windows into sequences, and give each source equal total metric weight.

Here we compare normal and final progressive encoders. The capstone selects a JEPA stage using validation macro-F1 before reading test performance. The original skeleton S-JEPA uses a teacher readout; our online readout is a declared adaptation.

In [ ]:
from aaai_gait.training import embed_windows
from aaai_gait.workflow import select_probe
comparisons=[]
for name,encoder in [('normal',anchor),('progressive',adapted)]:
    z=embed_windows(encoder,x,batch_size=settings.batch_size)
    probe,selection=select_probe(z,meta,settings.seed)
    comparisons.append({'stage':name,**selection})
display(pd.DataFrame(comparisons))

## 3. Keep anomaly detection anchored

After learning an abnormal pattern, a predictor may find it less surprising. That is useful for condition recognition but can weaken normal-only anomaly detection. The normal anchor remains a separate checkpoint and supplies the main anomaly score. The adapted encoder is evaluated as a supervised feature extractor. Do not describe its error as an untouched normality reference.

## 4. Decide when stochastic latent variables are warranted

Normal gait has several plausible continuations from a short prefix. A deterministic predictor can average incompatible futures. The 2026 stochastic-latent JEPA paper introduces a training posterior `q(z | context, target)`, a test-time prior `p(z | context)`, a predictor conditioned on `z`, and a KL penalty aligning posterior and prior. Sampling the prior can then produce diverse latent completions.

This tutorial implements uniform masking and the optional StoP-inspired positional-noise ablation. It does not implement that separate stochastic-latent architecture. Adding independent noise without a learned prior, posterior, and calibration experiment would not reproduce the paper. A defensible extension would compare expected predictive error, uncertainty-error correlation, diversity, and representation quality across matched compute budgets.

Keep this extension secondary for a near-deadline paper. The primary claims can be tested with the deterministic normal anchor.

![Stochasticity](images/stochasticity.svg)

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.